# Part 1 - Distributed Data Processing with Spark

In [134]:
from pathlib import Path
import requests
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
import time
import pandas as pd

In [135]:
# Define URL for required file
taxi_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"

# Create data/raw directory if it doesn't exist
BASE_DIR = Path.cwd().resolve()
data_dir = BASE_DIR / "data" / "raw"
data_dir.mkdir(parents=True, exist_ok=True)

# Defines File path for downloaded data
taxi_path = data_dir / "yellow_tripdata_2024-01.parquet"

# Download File and write to specified path
def download_file(url, path):
    if path.exists():
        return
     
    with requests.get(url, stream=True, timeout=30) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)

download_file(taxi_url, taxi_path)
print("\nFile downloaded successfully!")


File downloaded successfully!


In [136]:
# Creating a Spark Session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("LLM-Powered Applications & Distributed Computing") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# Verifying the session creation
print(f'Spark version: {spark.version}')
print(f'App name: {spark.sparkContext.appName}')
print(f'Master: {spark.sparkContext.master}')
print(f'Default parallelism: {spark.sparkContext.defaultParallelism}')

Spark version: 4.0.2
App name: LLM-Powered Applications & Distributed Computing
Master: local[*]
Default parallelism: 2


In [137]:
# Load the parquet file into a Spark Dataframe
df = spark.read.parquet(str(taxi_path))

# Display the schema: Column names and types
df.printSchema()

# Verify the first 5 rows of data
df.show(5, truncate=True)

# Display total number of rows
print(f"\nTotal number of rows: {df.count():,}")

# Display total number of columns
print(f"Total number of columns: {len(df.columns)}")

# Display total number of partitions
print(f"Total number of partitions: {df.rdd.getNumPartitions()}")

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 00:57:55|  2024-01-01 01:17:43|              1|         1.72|         1|                 N|         186|          79|           2|       17.7|  1.0|    0.5|       0.

In [138]:
# Timing for Reading Data with Spark
start = time.time()
df_spark = spark.read.parquet(str(taxi_path))
spark_read_time = time.time() - start

# Timing for Compution with Spark
start = time.time()
spark_count = df_spark.count()
spark_action_time = time.time() - start

# Timing for Reading Data with Pandas
start = time.time()
df_pandas = pd.read_parquet(str(taxi_path))
pandas_read_time = time.time() - start

# Display Results
print(f"Spark schema read (lazy): {spark_read_time:.3f}s")
print(f"Spark count action: {spark_action_time:.3f}s ({spark_count:,} rows)")
print(f"Pandas full read: {pandas_read_time:.3f}s ({len(df_pandas):,} rows)")
print(f"Pandas memory usage: {df_pandas.memory_usage(deep=True).sum() / 1e6:.1f} MB")

# Cleaning up Pandas Dataframe
del df_pandas

Spark schema read (lazy): 0.188s
Spark count action: 0.235s (2,964,624 rows)
Pandas full read: 2.460s (2,964,624 rows)
Pandas memory usage: 535.9 MB


The Spark load time is seen to be much faster because it is lazy meaning that just the metadata is being loaded with no actual data. Upon conducting an action using Spark, the data would then be loaded hence the time result seen for the Spark count action. In contrast, Pandas is much slower due to the full dataset being loaded into memory immediately.

In [139]:
# Remove rows with nulls in critical columns
def remove_nulls(df_spark):
    initial_count = df_spark.count()

    df_spark = df_spark.dropna(subset=[
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_distance",
        "PULocationID",
        "DOLocationID",
        "fare_amount"
    ])
    removed_count = initial_count - df_spark.count()
    print(f"Removed {removed_count:,} rows with null values.")
    return df_spark

# Filtering out invalid trips
def filter_trips(df_spark):
    initial_count = df_spark.count()

    df_spark = df_spark.filter(
        (F.col("trip_distance") > 0) &
        (F.col("fare_amount") > 0) &
        (F.col("fare_amount") <= 500) &
        (F.col("tpep_dropoff_datetime") >= F.col("tpep_pickup_datetime"))
    )
    removed_count = initial_count - df_spark.count()
    print(f"Removed {removed_count:,} rows with invalid trip data.")
    return df_spark

df_spark = remove_nulls(df_spark)
df_spark = filter_trips(df_spark)
print(f"\nRemaining rows after cleaning: {df_spark.count():,}")

Removed 0 rows with null values.
Removed 94,996 rows with invalid trip data.

Remaining rows after cleaning: 2,869,628


In [140]:
def create_derived_columns(df_spark):
    df_spark = df_spark.withColumns({
        "trip_duration_minutes": (
            F.unix_timestamp("tpep_dropoff_datetime") -
            F.unix_timestamp("tpep_pickup_datetime")
        ) / 60,

        "pickup_hour": F.hour("tpep_pickup_datetime"),

        "pickup_day_of_week": F.date_format("tpep_pickup_datetime", "EEEE"),

        "tip_percentage": F.when(
            F.col("fare_amount") > 0,
            (F.col("tip_amount") / F.col("fare_amount")) * 100
        ).otherwise(0)
    })

    df_spark = df_spark.withColumn(
        "trip_speed_mph",
        F.when(
            F.col("trip_duration_minutes") > 0,
            F.col("trip_distance") / (F.col("trip_duration_minutes") / 60)
        ).otherwise(0)
    )

    return df_spark

df_spark = create_derived_columns(df_spark)
print("\nDerived columns created successfully!\n")
df_spark.show(5, truncate=True)


Derived columns created successfully!

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+---------------------+-----------+------------------+------------------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|trip_duration_minutes|pickup_hour|pickup_day_of_week|    tip_percentage|    trip_speed_mph|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------

In [141]:
# Register as a temporary view for SQL queries
df_spark.createOrReplaceTempView("taxi_trips")

In [142]:
# Query 1: Top 10 Busiest Pickup Hours
busiest_hours = spark.sql("""
    SELECT
        pickup_hour,
        COUNT(*) AS trip_count,
        ROUND(AVG(fare_amount), 2) AS avg_fare,
        ROUND(AVG(tip_percentage), 2) AS avg_tip_percentage
    FROM taxi_trips
    GROUP BY pickup_hour
    ORDER BY trip_count DESC
    LIMIT 10
""")

print("Top 10 Busiest Pickup Hours:")
busiest_hours.show()

Top 10 Busiest Pickup Hours:


+-----------+----------+--------+------------------+
|pickup_hour|trip_count|avg_fare|avg_tip_percentage|
+-----------+----------+--------+------------------+
|         18|    206259|   17.02|             22.78|
|         17|    200285|   18.12|             22.34|
|         16|    184950|   19.46|             21.84|
|         15|    183976|   19.11|              19.8|
|         19|    178787|   17.63|             22.86|
|         14|    178007|   19.27|              19.8|
|         13|    165332|   18.42|             19.79|
|         12|    159888|    17.8|             19.74|
|         21|    155890|    18.3|             21.88|
|         20|    155541|   18.05|             22.17|
+-----------+----------+--------+------------------+



In [143]:
# Query 2: Day with Highest Average Trip Speed
fastest_day = spark.sql("""
    SELECT
        pickup_day_of_week,
        ROUND(AVG(trip_speed_mph), 2) AS avg_speed,
        ROUND(AVG(trip_distance), 2) AS avg_distance,
        ROUND(AVG(trip_duration_minutes), 2) AS avg_duration
    FROM taxi_trips
    GROUP BY pickup_day_of_week
    ORDER BY avg_speed DESC
    LIMIT 1
""")

print("Day with Highest Average Trip Speed:")
fastest_day.show()

Day with Highest Average Trip Speed:
+------------------+---------+------------+------------+
|pickup_day_of_week|avg_speed|avg_distance|avg_duration|
+------------------+---------+------------+------------+
|           Tuesday|    17.35|        4.25|       16.17|
+------------------+---------+------------+------------+



In [144]:
# Query 3: Top 5 Pickup Locations by Total Revenue for Each Day of the Week
top_locations = spark.sql("""
    SELECT *
    FROM (
        SELECT
            pickup_day_of_week,
            PULocationID,
            SUM(fare_amount) AS total_revenue,
            RANK() OVER (
                PARTITION BY pickup_day_of_week
                ORDER BY SUM(fare_amount) DESC
            ) AS rank
        FROM taxi_trips
        GROUP BY pickup_day_of_week, PULocationID
    ) ranked
    WHERE rank <= 5
    ORDER BY pickup_day_of_week, total_revenue DESC
""")

print("Top 5 Pickup Locations by Total Revenue for Each Day of the Week:")
top_locations.show(top_locations.count(), truncate=False)

Top 5 Pickup Locations by Total Revenue for Each Day of the Week:


+------------------+------------+------------------+----+
|pickup_day_of_week|PULocationID|total_revenue     |rank|
+------------------+------------+------------------+----+
|Friday            |132         |1076822.660000001 |1   |
|Friday            |138         |483833.14         |2   |
|Friday            |161         |272366.7200000002 |3   |
|Friday            |237         |248581.56999999992|4   |
|Friday            |236         |248299.26000000065|5   |
|Monday            |132         |1593427.760000003 |1   |
|Monday            |138         |648123.5200000003 |2   |
|Monday            |161         |299014.9200000002 |3   |
|Monday            |186         |236408.26000000077|4   |
|Monday            |236         |236288.67000000092|5   |
|Saturday          |132         |1003226.860000001 |1   |
|Saturday          |138         |304327.1499999993 |2   |
|Saturday          |230         |248186.18000000055|3   |
|Saturday          |249         |226590.65000000008|4   |
|Saturday     

In [145]:
# Query 3: Top 5 Pickup Locations by Total Revenue for Each Day of the Week
daily_location_revenue = df_spark.groupBy(
    "pickup_day_of_week", "PULocationID"
    ).agg(
        F.sum("fare_amount").alias("total_revenue")
    )

day_window = Window.partitionBy("pickup_day_of_week").orderBy(F.desc("total_revenue"))

ranked_locations = daily_location_revenue.withColumn(
    "rank", F.rank().over(day_window)
)

top_locations = ranked_locations.filter(F.col("rank") <= 5)

top_5_locations = top_locations.orderBy("pickup_day_of_week", F.desc("total_revenue"))

print("Top 5 Pickup Locations by Total Revenue for Each Day of the Week:")
top_5_locations.show(top_5_locations.count(), truncate=False)

Top 5 Pickup Locations by Total Revenue for Each Day of the Week:
+------------------+------------+------------------+----+
|pickup_day_of_week|PULocationID|total_revenue     |rank|
+------------------+------------+------------------+----+
|Friday            |132         |1076822.660000001 |1   |
|Friday            |138         |483833.14         |2   |
|Friday            |161         |272366.7200000002 |3   |
|Friday            |237         |248581.56999999992|4   |
|Friday            |236         |248299.26000000065|5   |
|Monday            |132         |1593427.760000003 |1   |
|Monday            |138         |648123.5200000003 |2   |
|Monday            |161         |299014.9200000002 |3   |
|Monday            |186         |236408.26000000077|4   |
|Monday            |236         |236288.67000000092|5   |
|Saturday          |132         |1003226.860000001 |1   |
|Saturday          |138         |304327.1499999993 |2   |
|Saturday          |230         |248186.18000000055|3   |
|Satur

In [146]:
# Query 4: Day when Cumulative Trip Count by Hour of the Day Surpasses 50% of Daily Trips
cumulative_trips_50 = spark.sql("""
    WITH hourly_counts AS (
        SELECT
            pickup_hour,
            COUNT(*) AS trips_per_hour
        FROM taxi_trips
        GROUP BY pickup_hour
    ),
    cumulative AS (
        SELECT
            pickup_hour,
            trips_per_hour,
            SUM(trips_per_hour) OVER (
                ORDER BY pickup_hour 
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS cumulative_trips,
            SUM(trips_per_hour) OVER () AS total_trips
        FROM hourly_counts
    ),
    first_surpassed AS (
        SELECT pickup_hour
        FROM cumulative
        WHERE cumulative_trips >= 0.5 * total_trips
        ORDER BY pickup_hour
        LIMIT 1
    )
    SELECT
        c.pickup_hour,
        c.trips_per_hour,
        c.cumulative_trips,
        ROUND(c.cumulative_trips / c.total_trips, 2) AS cumulative_fraction,
        CASE WHEN c.pickup_hour = f.pickup_hour THEN 1 ELSE 0 END AS surpassed_50_percent
    FROM cumulative c
    LEFT JOIN first_surpassed f
        ON c.pickup_hour = f.pickup_hour
    ORDER BY c.pickup_hour
""")

print("Cumulative Trip Count By Hour of the Day with Flag when 50% of Daily Trips is Surpassed:")
cumulative_trips_50.show(24)


Cumulative Trip Count By Hour of the Day with Flag when 50% of Daily Trips is Surpassed:
+-----------+--------------+----------------+-------------------+--------------------+
|pickup_hour|trips_per_hour|cumulative_trips|cumulative_fraction|surpassed_50_percent|
+-----------+--------------+----------------+-------------------+--------------------+
|          0|         75236|           75236|               0.03|                   0|
|          1|         50481|          125717|               0.04|                   0|
|          2|         34960|          160677|               0.06|                   0|
|          3|         22941|          183618|               0.06|                   0|
|          4|         15275|          198893|               0.07|                   0|
|          5|         17492|          216385|               0.08|                   0|
|          6|         39414|          255799|               0.09|                   0|
|          7|         80856|          336

In [147]:
# Query 5: Comparison of Average Fare, Distance an Tip Percentage between Short, Medium, and Long Trips
trip_comparison = spark.sql("""
    WITH categorized AS (
        SELECT
            CASE
                WHEN trip_distance < 2 THEN "Short (<2 miles)"
                WHEN trip_distance <= 10 THEN "Medium (2-10 miles)"
                ELSE "Long (>10 miles)"
            END AS distance_category,
            fare_amount,
            trip_distance,
            tip_percentage
        FROM taxi_trips
    ),
    aggregated AS (
        SELECT
            distance_category,
            COUNT(*) AS trip_count,
            ROUND(AVG(fare_amount), 2) AS avg_fare,
            ROUND(AVG(trip_distance), 2) AS avg_distance,
            ROUND(AVG(tip_percentage), 2) AS avg_tip_percentage
        FROM categorized
        GROUP BY distance_category
    )
    SELECT *,
        RANK() OVER (ORDER BY avg_tip_percentage DESC) AS tip_rank
    FROM aggregated
    ORDER BY avg_tip_percentage DESC
""")

print("Comparison of Average Fare, Distance and Tip Percentage between Short, Medium, and Long Trips:")
trip_comparison.show()

Comparison of Average Fare, Distance and Tip Percentage between Short, Medium, and Long Trips:
+-------------------+----------+--------+------------+------------------+--------+
|  distance_category|trip_count|avg_fare|avg_distance|avg_tip_percentage|tip_rank|
+-------------------+----------+--------+------------+------------------+--------+
|   Short (<2 miles)|   1642198|    9.91|        1.13|             23.07|       1|
|   Long (>10 miles)|    225017|   64.67|        21.7|             21.93|       2|
|Medium (2-10 miles)|   1002413|   22.18|        3.96|             18.57|       3|
+-------------------+----------+--------+------------+------------------+--------+

